# FR3 Duo handover · SmolVLA training on free Colab

Pipeline R1 → R4 on a free Colab GPU, then the model + model card to a **private** Hugging Face model repo.

**Rules**
- The dataset is **never** uploaded to the Hub. It comes from *your* Google Drive (organiser data, team use only).
- Registry and outputs live on Drive, so they survive a disconnect. Checkpoints too → training can resume.
- A **human** approves the dataset (step 6). Numbers only from logged runs.

**Before the first run** — put the datasets in `MyDrive/shs-hack-2026/data/` (from the organisers' file share),
as zips or extracted folders (each with `meta/`, `data/`, `videos/` inside), at any depth. **Drive has 15 GB:**
keep each dataset **once, as a zip** (~4.4 GB for all three) — step 4 unzips to Colab's local disk anyway, and
training keeps only the last 2 checkpoints on Drive (`keep_checkpoints` in the train config). Step 4 finds all of
them; a dataset added later is picked up by re-running steps 4 and 6 (or **Data → Add demonstrations** in the
console, which can also fetch from Hugging Face, GitHub, a Drive link or a cloud bucket).
Add a Colab secret `HF_TOKEN` (🔑 icon, write access) — only needed for step 10.

**Runtime → Change runtime type → T4 GPU.**

## 0 · Settings

In [ ]:
REPO_URL = "https://github.com/RohithThota97/shs-hack-2026-vla-pipeline.git"
BRANCH = "main"
DRIVE_DIR = "/content/drive/MyDrive/shs-hack-2026"       # data zips in DRIVE_DIR/data, state in DRIVE_DIR/state
# Every dataset under DRIVE_DIR/data (folder with meta/info.json, or a zip of one) is found and linked as
# data/raw/<folder name> — drop a new one on Drive and re-run step 4. These two keep their earlier names because
# demo IDs depend on data/raw/<name> (ds-v1 was built with them): data/raw/<name> ← (episode count, zip name).
# Keep in sync with dataset.pinned_names in configs/project.yaml (the console uses that list).
PINNED = {
    "fr3_duo_handover_84": (84, "fr3_duo_handover_84_ep.zip"),
    "fr3_duo_handover_lighting10": (10, "FR3Duo_Handover_Dataset_lerobot3.zip"),
}
LEROBOT_SPEC = "lerobot[smolvla,dataset]==0.6.1"   # datasets were written with 0.6.1; [dataset] pins av<16 + torchcodec
TRAIN_CONFIG = "configs/train/smolvla_fr3_duo.yaml"
HF_MODEL_REPO = "RohithThota/smolvla-fr3duo-handover"   # private model repo
REPO = "/content/repo"

## 1 · GPU, Drive, code

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount("/content/drive")
import os, subprocess
os.makedirs(f"{DRIVE_DIR}/data", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/state/registry", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/state/outputs/logs", exist_ok=True)
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
%cd {REPO}
!git log --oneline -1

## 2 · Install

LeRobot may install a different torch than Colab ships. If the output says to **restart the session**, do it
(Runtime → Restart session) and re-run cells 0 and 1, then continue at step 3.

In [ ]:
!pip install -q -e ".[dev,video,hub,console]" "{LEROBOT_SPEC}" matplotlib
# Colab preinstalls diffusers + an old torchao; LeRobot imports diffusers (via GR00T) at start-up and that old
# torchao breaks it ("cannot import name 'FqnToConfig'"). SmolVLA doesn't use torchao, so remove it.
!pip uninstall -y -q torchao
!lerobot-train --help > /dev/null && echo "lerobot-train starts OK"
!pip freeze | grep -i -E "^(lerobot|torch|torchvision|torchcodec|av|huggingface.hub)=="
import av; assert av.__version__.startswith("15."), f"PyAV {av.__version__}: LeRobot 0.6.1 needs av>=15,<16 — restart the session"

## 3 · Persistent state on Drive

`registry/` and `outputs/` in the repo become links to Drive. Demo IDs depend on `data/raw/<name>`, so they are
the same on every Colab session (and on a laptop with the same layout).

In [ ]:
import shutil
for name in ("registry", "outputs"):
    p = f"{REPO}/{name}"
    if os.path.islink(p):
        continue
    shutil.rmtree(p, ignore_errors=True)
    os.symlink(f"{DRIVE_DIR}/state/{name}", p)
!ls -la registry outputs

## 3b · Console (optional) — the web UI on this GPU machine

Starts `hackpipe console` and a free `cloudflared` quick tunnel, and prints a link. The link is public, so the
console asks for a fresh access token (already in the printed link). Share it only with the team; stop the tunnel
when done. Train → Timing run / Start training run as background jobs on this GPU; their logs and checkpoints are
on Drive. The registry writer is "colab" here — release it before writing from the laptop.

In [ ]:
import re, secrets, subprocess, time
if not os.path.exists("/content/cloudflared"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
    !chmod +x /content/cloudflared
TOKEN = secrets.token_urlsafe(16)
env = {**os.environ, "HACKPIPE_CONSOLE_TOKEN": TOKEN}
console = subprocess.Popen(["hackpipe", "console", "--port", "8501"], cwd=REPO, env=env,
                           stdout=open("/content/console.log", "w"), stderr=subprocess.STDOUT)
tunnel = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", "http://localhost:8501", "--no-autoupdate"],
                          stdout=open("/content/tunnel.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/tunnel.log").read())
    if m:
        print("Console:", f"{m.group(0)}/?token={TOKEN}")
        break
    time.sleep(1)
else:
    print("No tunnel URL yet — see /content/tunnel.log")

In [ ]:
# Stop the console and the tunnel
console.terminate(); tunnel.terminate()

## 4 · Data: Drive → local disk (fast), linked into `data/raw/`

Finds **every** dataset under `DRIVE_DIR/data`: extracted folders (any depth, e.g. `fr3_duo_rand500/fr3_duo_rand500`)
and zips that aren't extracted there. Each becomes `data/raw/<folder name>`; the pinned ones keep their names
(matched by episode count / zip name). A copy is only reused if it finished (`.complete` marker). New dataset on
Drive → just re-run this cell and step 6. If two folders get the same name, set `SRC` by hand.

In [ ]:
import glob, json, shutil, zipfile
from hackpipe.pipeline.sources import dataset_name, extract_zip, find_roots
SRC = {}   # optional, by hand: {"<name>": "/content/drive/MyDrive/.../<folder or zip>"} — replaces what is found
by_eps = {n: k for k, (n, _) in PINNED.items()}
by_zip = {z: k for k, (_, z) in PINNED.items()}
found = {}
for root in find_roots(f"{DRIVE_DIR}/data"):
    n = json.load(open(f"{root}/meta/info.json"))["total_episodes"]
    name = by_eps.get(n) or dataset_name(root)
    if name in found:
        raise RuntimeError(f"{found[name]} and {root} both map to data/raw/{name} — set SRC by hand")
    found[name] = str(root)
    print(f"found folder: {n} episodes ← {root} → data/raw/{name}")
for z in sorted(glob.glob(f"{DRIVE_DIR}/data/**/*.zip", recursive=True)):
    name = by_zip.get(os.path.basename(z)) or dataset_name(z)
    if name not in found:                                    # zips whose folder is already there are skipped
        found[name] = z
        print(f"found zip: {z} → data/raw/{name}")
found.update(SRC)

os.makedirs(f"{REPO}/data/raw", exist_ok=True)
READY = []
for name, src in sorted(found.items()):
    dest = f"/content/data_cache/{name}"
    if not os.path.exists(f"{dest}/.complete"):
        shutil.rmtree(dest, ignore_errors=True)               # half-finished copy from a dropped session
        print(f"{'unzipping' if src.endswith('.zip') else 'copying'} {name} from Drive …")
        if src.endswith(".zip"):
            os.makedirs(dest); extract_zip(src, dest)
        else:
            shutil.copytree(src, dest)
        open(f"{dest}/.complete", "w").close()
    roots = find_roots(dest)
    if not roots:
        print(f"skip {name}: no meta/info.json inside {src}")
        continue
    for root in roots:                                        # a zip may hold several datasets
        link_name = name if len(roots) == 1 else f"{name}__{dataset_name(root)}"
        link = f"{REPO}/data/raw/{link_name}"
        if os.path.islink(link): os.unlink(link)
        os.symlink(root, link)
        info = json.load(open(f"{root}/meta/info.json"))
        print(f"ready data/raw/{link_name}: {info['total_episodes']} episodes, {info['total_frames']:,} frames, "
              f"{info['fps']} fps ← {src}")
        READY.append(link_name)
print("datasets:", READY)

## 5 · AV1 decode check (likely blocker)

All videos are AV1. This loads samples through LeRobot itself — the same path training uses — and times reads.
If it fails, try `VIDEO_BACKEND = "pyav"`; if that fails too, use the re-encode cell below (creates a new dataset
version, because the video files change).

In [ ]:
import time, random
from lerobot.datasets.lerobot_dataset import LeRobotDataset   # path differs in some LeRobot versions
VIDEO_BACKEND = None    # None = LeRobot default; or "pyav" / "torchcodec"
kw = {"video_backend": VIDEO_BACKEND} if VIDEO_BACKEND else {}
CHECK = READY[0]        # or any name from step 4
ds = LeRobotDataset(f"local/{CHECK}", root=f"{REPO}/data/raw/{CHECK}", **kw)
x = ds[0]
print({k: tuple(v.shape) for k, v in x.items() if hasattr(v, "shape")})
t0 = time.time()
for i in random.sample(range(len(ds)), 100):
    ds[i]
print(f"{100 / (time.time() - t0):.1f} samples/s (single process) — training needs ≥ batch_size / s_per_step")

In [ ]:
# ONLY if decoding failed: re-encode the videos to h264 into a new folder, then point data/raw at it.
# !lerobot-edit-dataset --repo_id local/fr3_duo_handover_84 --root data/raw/fr3_duo_handover_84 \
#     --new_root /content/data_cache/fr3_duo_handover_84_h264 \
#     --operation.type reencode_videos --operation.rgb_encoder.vcodec h264

## 6 · R1–R3: ingest, quality check, dataset version

Runs on every dataset found in step 4. Already registered demos with unchanged files are skipped, so re-running
after adding a dataset only imports and checks the new one.

In [ ]:
!pytest -q 2>&1 | tail -2
for name in READY:
    !hackpipe ingest --root data/raw/{name}
    !hackpipe qc --root data/raw/{name} | head -20
!hackpipe dataset list

In [ ]:
# Creates the next draft version (ds-v1 on the first run; later ones build on the last approved version).
# Skip if `dataset list` already shows the one you need. Same in one step: !hackpipe auto --all
NOTES = "v1: night84 90/10 train/val; test = lighting10 (held-out source)"
!hackpipe dataset create --notes "{NOTES}"

In [ ]:
DS = "ds-v1"     # the version printed above
from IPython.display import Markdown, display
from hackpipe.quality.report import report_name
for name in READY:
    qc = f"outputs/qc/{report_name(f'data/raw/{name}')}/qc_report.md"
    if os.path.exists(qc):
        display(Markdown(open(qc).read()))
display(Markdown(open(f"outputs/datasets/{DS}/dataset_card.md").read()))

### ⛔ STOP — human approval

Read the QC report and the dataset card above. If they are right, put your name below and run the cell.
Agents never run this cell.

In [ ]:
APPROVED_BY = ""        # your name
assert APPROVED_BY, "A human must review and approve the dataset"
!hackpipe dataset approve {DS} --by "{APPROVED_BY}"
!hackpipe status

## 7 · Dry run — check the `lerobot-train` command line by line

In [ ]:
!hackpipe train --config {TRAIN_CONFIG} --dataset {DS} --dry-run

## 8 · Timing run (~15 min) → seconds/step and peak VRAM

The timing run gets a model ID too: note it as "timing run, not a result". Out of memory → set
`EXTRA = "--set batch_size=4"` and repeat (and use the same batch size for the full run).

In [ ]:
EXTRA = ""
import threading, time
vram, stop = [], threading.Event()
def poll():
    while not stop.is_set():
        out = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                             capture_output=True, text=True).stdout.strip()
        if out: vram.append(int(out.splitlines()[0]))
        time.sleep(5)
th = threading.Thread(target=poll, daemon=True); th.start()
!hackpipe train --config {TRAIN_CONFIG} --dataset {DS} --set steps=200 --set save_freq=200 {EXTRA} 2>&1 | tee outputs/logs/train_timing.log
stop.set()
print("peak VRAM MiB:", max(vram) if vram else "n/a")
!grep -E "step:" outputs/logs/train_timing.log | tail -5

## 9 · Full training run

Set the step budget from the timing run: `steps = slot_seconds / seconds_per_step`, rounded down to a multiple of
`save_freq` (1000). Free Colab sessions end after a few hours — checkpoints every 1000 steps are on Drive, and the
model is registered when training starts, so a dropped session can resume.

In [ ]:
SEC_PER_STEP = None     # from the timing log (updt_s + data_s per step)
SLOT_HOURS = 3.5        # what you expect Colab to give you in one session
STEPS = int(SLOT_HOURS * 3600 / SEC_PER_STEP) // 1000 * 1000
print("steps:", STEPS, "→ record the reason in docs/decisions.md and commit the value in", TRAIN_CONFIG)

In [ ]:
!hackpipe train --config {TRAIN_CONFIG} --dataset {DS} --set steps={STEPS} {EXTRA} 2>&1 | tee -a outputs/logs/train_v1.log

In [ ]:
# After a disconnect: re-run cells 0–4, then resume with the model ID printed when training started.
# !hackpipe train --config {TRAIN_CONFIG} --dataset {DS} --resume <model_id> 2>&1 | tee -a outputs/logs/train_v1.log

In [ ]:
MODEL = ""     # the model ID of the full run (printed when training started)
!python scripts/plot_loss.py --log outputs/logs/train_v1.log --model {MODEL}
!hackpipe status
from IPython.display import Image
Image(f"outputs/runs/{MODEL}/loss.png")

## 10 · Model + model card → Hugging Face (private)

Uploads only the checkpoint, `README.md` (model card), run record, dataset summary (IDs and counts, no data)
and the loss curve. Check the dry run first.

In [ ]:
from google.colab import userdata
from IPython.display import Markdown, display
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
assert HF_MODEL_REPO, "Set HF_MODEL_REPO in step 0"
!hackpipe publish --model {MODEL} --repo {HF_MODEL_REPO} --dry-run
display(Markdown(open(f"outputs/hub/{MODEL}/README.md").read()))

In [ ]:
!hackpipe publish --model {MODEL} --repo {HF_MODEL_REPO}